# 04 -- XGBoost Black Hole Mass Prediction

Predicts quasar supermassive black hole mass (`LOGMBH`) from DRW variability
parameters and observational properties using an XGBoost gradient-boosted tree.

**Features:** `log_tau`, `log_sigma`, `mu`, `PSFMAG_r`, `Z_FIT`
**Target:**   `LOGMBH` (log base-10 solar masses)

**Input:**  `data/DRW_results.csv` (output of notebook 03)
**Outputs:** Diagnostic plots (predicted vs true, feature importance, residuals)

This notebook trains and evaluates a single XGBoost model. For a full
multi-model comparison and ablation study, see notebook 05.


In [ ]:
# !pip install xgboost scikit-learn matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

DATA_DIR = Path("data")

## 1. Load Data and Prepare Features

In [ ]:
DATA_PATH = DATA_DIR / "DRW_results.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows")
print(df[["tau", "sigma", "mu", "PSFMAG_r", "Z_FIT", "LOGMBH"]].describe())


## 2. Feature Engineering and Cleaning

`tau` and `sigma` span orders of magnitude, so we work in log-space.
We drop any rows where the DRW fit produced non-physical values (tau <= 0)
and verify there are no remaining infinities or NaNs before training.


In [ ]:
FEATURES = ["tau", "sigma", "mu", "PSFMAG_r", "Z_FIT"]
TARGET   = "LOGMBH"

# Drop rows with missing values in features or target
df_clean = df.dropna(subset=FEATURES + [TARGET]).copy()
print(f"Rows after NaN drop: {len(df_clean)}")

# Remove non-physical DRW fits (tau and sigma must be positive)
bad = (df_clean["tau"] <= 0) | (df_clean["sigma"] <= 0)
print(f"Rows with tau<=0 or sigma<=0: {bad.sum()}")
df_clean = df_clean[~bad].copy()

# Log-transform -- tau and sigma span orders of magnitude
df_clean["log_tau"]   = np.log10(df_clean["tau"])
df_clean["log_sigma"] = np.log10(df_clean["sigma"])

FEATURES_TRANSFORMED = ["log_tau", "log_sigma", "mu", "PSFMAG_r", "Z_FIT"]

# Check for any infinities introduced by the log transform
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)
bad_after_log = df_clean[FEATURES_TRANSFORMED].isna().any(axis=1)
print(f"Rows with inf/nan after log transform: {bad_after_log.sum()}")
df_clean = df_clean[~bad_after_log].copy()

X = df_clean[FEATURES_TRANSFORMED].values
y = df_clean[TARGET].values

print(f"\nFinal sample: {len(df_clean)} quasars")
print(f"X shape: {X.shape}")
print(f"y range: {y.min():.2f} -- {y.max():.2f}")


## 3. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")


## 4. Train XGBoost

We use a shallow tree (max_depth=3) with slow learning rate and early stopping
to reduce overfitting on this relatively small astrophysical dataset.


In [ ]:
model = XGBRegressor(
    n_estimators       = 2000,
    learning_rate      = 0.01,
    max_depth          = 3,
    subsample          = 0.8,
    colsample_bytree   = 0.8,
    reg_alpha          = 0.2,
    reg_lambda         = 1.5,
    eval_metric        = "rmse",
    early_stopping_rounds = 30,
    random_state       = 42,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)


## 5. Evaluate on Hold-Out Test Set

In [ ]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"Test RMSE : {rmse:.4f} dex")
print(f"Test R2   : {r2:.4f}")


## 6. 5-Fold Cross-Validation

In [ ]:
cv_model = XGBRegressor(
    n_estimators   = 500,
    learning_rate  = 0.05,
    max_depth      = 4,
    subsample      = 0.8,
    colsample_bytree = 0.8,
    random_state   = 42,
)

kf      = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2   = cross_val_score(cv_model, X, y, cv=kf, scoring="r2")
cv_rmse = cross_val_score(cv_model, X, y, cv=kf, scoring="neg_root_mean_squared_error")

print(f"5-Fold CV R2  : {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}")
print(f"5-Fold CV RMSE: {(-cv_rmse).mean():.4f} +/- {(-cv_rmse).std():.4f} dex")


## 7. Predicted vs True Black Hole Mass

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(y_test, y_pred, alpha=0.3, s=10, color="steelblue")

lims = [min(y_test.min(), y_pred.min()) - 0.2,
        max(y_test.max(), y_pred.max()) + 0.2]
ax.plot(lims, lims, "r--", lw=1.5, label="1:1")

ax.set_xlabel("True log(MBH/Msun)", fontsize=13)
ax.set_ylabel("Predicted log(MBH/Msun)", fontsize=13)
ax.set_title(f"XGBoost  |  RMSE={rmse:.3f} dex  |  R2={r2:.3f}", fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig(DATA_DIR / "predicted_vs_true.png", dpi=150)
plt.show()


## 8. Feature Importance

In [ ]:
importances = model.feature_importances_

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(FEATURES_TRANSFORMED, importances, color="steelblue")
ax.set_xlabel("Importance")
ax.set_title("XGBoost Feature Importance")
plt.tight_layout()
plt.savefig(DATA_DIR / "feature_importance.png", dpi=150)
plt.show()


## 9. Residual Plot

Residuals should be symmetric around zero with no obvious trend vs predicted mass.

In [ ]:
residuals = y_test - y_pred

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(y_pred, residuals, alpha=0.3, s=10, color="steelblue")
ax.axhline(0, color="red", linestyle="--", lw=1.5)
ax.set_xlabel("Predicted log(MBH/Msun)", fontsize=13)
ax.set_ylabel("Residual (True - Predicted)", fontsize=13)
ax.set_title("Residuals", fontsize=12)
plt.tight_layout()
plt.savefig(DATA_DIR / "residuals.png", dpi=150)
plt.show()
